# CA2 variant with individual-return factor inputs

Research code from my MSc thesis, University of Copenhagen, 2024.

Original research cells are preserved; saved outputs are cleared. See the repository README and docs/source_notes.md for inputs and execution context.


**Defining the utils**

**COMMENT: restart after having run this cell, so it is in memory**

**sliding window with validation**

In [ ]:
%%writefile utils.py
#!/usr/bin/env python
# -*- coding: utf-8 -*-
__author__ = 'Hans Andrias Gregoriussen'

import numpy as np

np.random.seed(42)


def format_time(t):
    """Return a formatted time string 'HH:MM:SS
    based on a numeric time() value"""
    m, s = divmod(t, 60)
    h, m = divmod(m, 60)
    return f'{h:0>2.0f}:{m:0>2.0f}:{s:0>2.0f}'

class MultipleTimeSeriesCV:
    """Generates tuples of train_idx, val_idx and test_idx pairs
    Assumes the MultiIndex contains levels 'symbol' and 'date'
    purges overlapping outcomes"""

    def __init__(self,
                 n_splits=3,
                 train_period_length=126,
                 test_period_length=21,
                 val_period_length=21,
                 lookahead=None,
                 date_idx='date',
                 shuffle=False):
        self.n_splits = n_splits
        self.lookahead = lookahead
        if self.lookahead is None:
            self.lookahead = 0
        self.test_length = test_period_length
        self.val_length = val_period_length
        self.train_length = train_period_length
        self.shuffle = shuffle
        self.date_idx = date_idx

    def split(self, X, y=None, groups=None):
        unique_dates = X.index.get_level_values(self.date_idx).unique()
        days = sorted(unique_dates, reverse=True)
        split_idx = []
        for i in range(self.n_splits):
            test_end_idx = i * self.test_length
            test_start_idx = test_end_idx + self.test_length
            val_end_idx = test_start_idx + self.lookahead
            val_start_idx = val_end_idx + self.val_length + self.lookahead
            train_end_idx = val_start_idx + self.lookahead
            train_start_idx = train_end_idx + self.train_length + self.lookahead
            split_idx.append([train_start_idx, train_end_idx,
                              val_start_idx, val_end_idx,
                              test_start_idx, test_end_idx])

        dates = X.reset_index()[[self.date_idx]]
        for train_start, train_end, val_start, val_end, test_start, test_end in split_idx:

            train_idx = dates[(dates[self.date_idx] > days[train_start])
                              & (dates[self.date_idx] <= days[train_end])].index
            val_idx = dates[(dates[self.date_idx] > days[val_start])
                              & (dates[self.date_idx] <= days[val_end])].index
            test_idx = dates[(dates[self.date_idx] > days[test_start])
                             & (dates[self.date_idx] <= days[test_end])].index
            if self.shuffle:
                np.random.shuffle(list(train_idx))
            yield train_idx.to_numpy(),val_idx.to_numpy(), test_idx.to_numpy()

    def get_n_splits(self, X, y, groups=None):
        return self.n_splits



class DateTimeSeriesCV:
    """Generates tuples of train_idx,val_idx and test_idx pairs
    Assumes the index is of datetime type
    purges overlapping outcomes"""

    def __init__(self, n_splits=3, train_period_length=126, val_period_length=21, test_period_length=21, lookahead=None, shuffle=False):
        self.n_splits = n_splits
        self.lookahead = lookahead
        if self.lookahead is None:
            self.lookahead = 0
        self.test_length = test_period_length
        self.val_length = val_period_length
        self.train_length = train_period_length
        self.shuffle = shuffle

    def split(self, X, y=None, groups=None):
        unique_dates = X.index.unique()
        days = sorted(unique_dates, reverse=True)

        split_idx = []
        for i in range(self.n_splits):
            test_end_idx = i * self.test_length
            test_start_idx = test_end_idx + self.test_length
            val_end_idx = test_start_idx + self.lookahead
            val_start_idx = val_end_idx + self.val_length + self.lookahead
            train_end_idx = val_start_idx + self.lookahead
            train_start_idx = train_end_idx + self.train_length + self.lookahead

            if train_start_idx >= len(days):
                break  # Stop if we've gone beyond the dataset

            split_idx.append([train_start_idx, train_end_idx, val_start_idx, val_end_idx, test_start_idx, test_end_idx])

        for train_start, train_end, val_start, val_end, test_start, test_end in split_idx:
            train_idx = X.index[(X.index > days[train_start]) & (X.index <= days[train_end])]
            val_idx = X.index[(X.index > days[val_start]) & (X.index <= days[val_end])]
            test_idx = X.index[(X.index > days[test_start]) & (X.index <= days[test_end])]

            if self.shuffle:
                train_idx = np.random.permutation(train_idx)

            yield train_idx, val_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys, os
from time import time
from pathlib import Path
from itertools import product
from tqdm import tqdm

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import time

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dot, Reshape, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import quantile_transform
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

from scipy.stats import spearmanr

from utils import format_time, MultipleTimeSeriesCV
from utils import format_time, DateTimeSeriesCV

from numpy.linalg import inv


**defining my results path**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/REVISED_results_CA_2')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)


**load beta input data**

In [ ]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')

file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta_input_revised_unbalanced.parquet'

# Make sure merged_df1 is a DataFrame and not a function or any other type.
merged_df_padded = pd.read_parquet(file_path)
print(merged_df_padded)

In [ ]:
import pandas as pd

# Function to calculate the ratio of missing to non-missing ret_excess for each date
def calculate_missing_target_ratio(df):
    # Group by date and calculate the number of missing and non-missing values for ret_excess
    missing_counts = df['ret_excess'].isna().groupby(level='date').sum()
    total_counts = df['ret_excess'].groupby(level='date').count() + missing_counts

    # Calculate the ratio of missing to non-missing values
    ratio_missing_to_non_missing = missing_counts / total_counts

    # Create a DataFrame to store the results
    ratio_df = pd.DataFrame({
        'missing_count': missing_counts,
        'total_count': total_counts,
        'ratio_missing_to_non_missing': ratio_missing_to_non_missing
    })

    return ratio_df

# Calculate the missing target ratio
ratio_df = calculate_missing_target_ratio(merged_df_padded)

# Print the result
print(ratio_df.mean())

import pandas as pd

def calculate_average_stocks_per_date(df):
    # Group by date and count the number of unique permnos (stocks) for each date
    stocks_per_date = df.groupby(level='date').size()

    # Calculate the average number of stocks present across all dates
    average_stocks_per_date = stocks_per_date.mean()

    return average_stocks_per_date

# Calculate the average number of stocks per date
average_stocks = calculate_average_stocks_per_date(merged_df_padded)

# Print the result
print(f"Average number of stocks present per date: {average_stocks:.2f}")
import pandas as pd

import pandas as pd

def calculate_average_stocks_not_present_per_date(df):
    # Get the total number of unique stocks (permnos) across the entire dataset
    total_unique_stocks = df.index.get_level_values('permno').unique().size
    print(f"Total unique stocks: {total_unique_stocks}")

    # Group by date and count the number of unique permnos (stocks) for each date
    stocks_per_date = df.groupby(level='date').size()
    print(f"Stocks per date:\n{stocks_per_date.head()}")

    # Calculate the number of stocks not present for each date
    stocks_not_present_per_date = total_unique_stocks - stocks_per_date
    print(f"Stocks not present per date:\n{stocks_not_present_per_date.head()}")

    # Calculate the average number of stocks not present across all dates
    average_stocks_not_present_per_date = stocks_not_present_per_date.mean()

    return average_stocks_not_present_per_date

# Assuming merged_df_padded is your DataFrame
# Calculate the average number of stocks not present per date
average_stocks_not_present = calculate_average_stocks_not_present_per_date(merged_df_padded)

# Print the result
print(f"Average number of stocks not present per date: {average_stocks_not_present:.2f}")



In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Group by date and calculate the number of observations for each characteristic
nobs_by_date = merged_df_padded.groupby(level='date').count().max(1)
narratives = [col for col in merged_df_padded.columns if col not in ['date', 'permno']] # Adjust this to exclude non-narrative columns
nobs_by_characteristic = merged_df_padded[narratives].groupby(level='date').count()

# Convert to long format for the heatmap
nobs_by_characteristic_long = nobs_by_characteristic.reset_index().melt(id_vars='date', var_name='Characteristic', value_name='# Observations')

# Create the plots
with sns.axes_style("white"):
    fig, axes = plt.subplots(ncols=2, figsize=(18, 8))

    # Distribution plot of number of stocks per month
    sns.histplot(nobs_by_date, kde=False, ax=axes[0])
    axes[0].set_title('# of Stocks per Month')
    axes[0].set_xlabel('# of Observations')

    # Heatmap of observations per stock characteristic
    heatmap_data = nobs_by_characteristic.T
    sns.heatmap(heatmap_data, ax=axes[1], cmap='Blues', cbar_kws={'label': '# Observations'})
    axes[1].set_title('# of Observations per Stock Characteristic')
    axes[1].set_xlabel('Date')
    axes[1].set_ylabel('Characteristic')
    plt.xticks(rotation=25, ha='right')

    sns.despine()
    fig.tight_layout()
    plt.show()


In [ ]:
nobs_by_date = merged_df_padded.groupby(level='date').count().max(1)
nobs_by_characteristic = pd.melt(merged_df_padded[narratives].groupby(level='date').count(),
                                 value_name='# Observations',
                                 var_name=['Characteristic'])
with sns.axes_style("white"):
    fig, axes = plt.subplots(ncols=2, figsize=(14, 4))
    sns.distplot(nobs_by_date, kde=False, ax=axes[0])
    axes[0].set_title('# of Stocks per Month')
    axes[0].set_xlabel('# of Observations')
    sns.boxplot(x='Characteristic',
                y='# Observations',
                data=nobs_by_characteristic,
                ax=axes[1],
                palette='Blues')
    axes[1].set_xticklabels(axes[1].get_xticklabels(),
                            rotation=25,
                            ha='right')
    axes[1].set_title('# of Observation per Stock Characteristic')
    sns.despine()
    fig.tight_layout()

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Reset index to access 'date' level
df_reset = merged_df_padded.reset_index()

# Extract year from the 'date' level and add as a new column
df_reset['year'] = df_reset['date'].dt.year

# Exclude observations for the year 2020
df_reset = df_reset[df_reset['year'] != 2020]

# Group by year and calculate the average number of observations per characteristic
narratives = [col for col in df_reset.columns if col not in ['date', 'permno', 'year']] # Adjust this to exclude non-narrative columns
nobs_by_year = df_reset.groupby('year')[narratives].count().mean(axis=1)

# Plot the average number of observations per year
with sns.axes_style("white"):
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.barplot(x=nobs_by_year.index, y=nobs_by_year.values, ax=ax, palette='Blues')

    # Display every second year on the x-axis
    ax.set_xticks(ax.get_xticks()[::2])

    ax.set_title('Average Number of Observations per Year')
    ax.set_xlabel('Year')
    ax.set_ylabel('Average # of Observations')
    sns.despine()
    fig.tight_layout()
    plt.show()


**MODEL**

In [ ]:
x_t_all_months = merged_df_padded

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade", "equal_weighted_portfolio"]

# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes

# Filter out 'ret_excess'
filtered_codes = [code for code in all_codes if code != "ret_excess"]

# Create narratives
narratives = [f'characteristic_{code}' for code in filtered_codes]

print(merged_df_padded.index)

n_factors = 6
n_narratives = len(narratives)
n_tickers = len(merged_df_padded.index.unique('permno'))

print(n_narratives)
print(n_tickers)

gpu_devices = tf.config.experimental.list_physical_devices('GPU')
if gpu_devices:
    print('Using GPU')
    tf.config.experimental.set_memory_growth(gpu_devices[0], True)
else:
    print('Using CPU')

**with regularization**

**CA_0**

In [ ]:
from tensorflow.keras.layers import Input, Dense, Dot
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1

def make_model(n_factors=6, return_components=False, l1_reg=0.01):
    input_beta = Input((n_tickers, n_narratives), name='input_beta')
    input_factor = Input((n_narratives,), name='input_factor')

    # Directly connect input_beta to output_beta with no hidden layer and no batch normalization
    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(input_beta)

    # Apply L1 regularization to the output_factor layer
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    output = Dot(axes=(2,1), name='output_layer')([output_beta, output_factor])

    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss='mse', optimizer='adam')

    # Create a standalone factor network model
    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model  # Return both the combined model and the factor network
    else:
        return model  # Return only the combined model by default

# To get both the main model and the factor network model
main_model, factor_network_model = make_model(n_factors=6, return_components=True, l1_reg=0.001)

# To get only the main model
main_model = make_model(n_factors=6, return_components=False, l1_reg=0.01)

main_model.summary()
factor_network_model.summary()


**CA_1**

In [ ]:
from tensorflow.keras.regularizers import l1

def make_model(n_factors=6, return_components=False, l1_reg=0.01):
    input_beta = Input((n_tickers, n_narratives), name='input_beta')
    input_factor = Input((n_narratives,), name='input_factor')

    hidden_layer = Dense(units=32, activation='relu', name='hidden_layer', kernel_regularizer=l1(l1_reg))(input_beta)
    batch_norm = BatchNormalization(name='batch_norm')(hidden_layer)

    # Apply L1 regularization to the output_beta layer
    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(batch_norm)

    # Apply L1 regularization to the output_factor layer
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    output = Dot(axes=(2,1), name='output_layer')([output_beta, output_factor])

    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss='mse', optimizer='adam')

    # Create a standalone factor network model
    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model  # Return both the combined model and the factor network
    else:
        return model  # Return only the combined model by default

# To get both the main model and the factor network model
main_model, factor_network_model = make_model(n_factors=6, return_components=True, l1_reg=0.001)

# To get only the main model
main_model = make_model(n_factors=6, return_components=False, l1_reg=0.01)

main_model.summary()
factor_network_model.summary()

**ca_2**

In [ ]:
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dot
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1



def make_model(n_tickers=20000,n_factors=6, return_components=False, l1_reg=0.01):
    input_beta = Input((n_tickers, n_narratives), name='input_beta')
    input_factor = Input((n_tickers,), name='input_factor')

    # First hidden layer with 32 neurons and batch normalization
    hidden_layer_1 = Dense(units=32, activation='relu', name='hidden_layer_1', kernel_regularizer=l1(l1_reg))(input_beta)
    batch_norm_1 = BatchNormalization(name='batch_norm_1')(hidden_layer_1)

    # Second hidden layer with 16 neurons and batch normalization
    hidden_layer_2 = Dense(units=16, activation='relu', name='hidden_layer_2', kernel_regularizer=l1(l1_reg))(batch_norm_1)
    batch_norm_2 = BatchNormalization(name='batch_norm_2')(hidden_layer_2)

    # Output beta layer with L1 regularization
    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(batch_norm_2)

    # Output factor layer with L1 regularization
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    # Dot product layer combining output_beta and output_factor
    output = Dot(axes=(2, 1), name='output_layer')([output_beta, output_factor])

    # Compile the main model
    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss='mse', optimizer='adam')

    # Create a standalone factor network model
    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model  # Return both the combined model and the factor network
    else:
        return model  # Return only the combined model by default


# To get both the main model and the factor network model
main_model, factor_network_model = make_model(n_factors=6, return_components=True, l1_reg=0.001)

# To get only the main model
main_model = make_model(n_factors=6, return_components=False, l1_reg=0.01)

# Print the summaries of the models
main_model.summary()
factor_network_model.summary()


**updated preprocessing code**

In [ ]:
import pandas as pd

def fill_with_cross_sectional_median(df, narratives):
    medians = df.groupby(level='date')[narratives].transform('median')
    return df.fillna(medians)

def fill_target_with_median(df):
    median_target = df.groupby(level='date')['ret_excess'].transform('median')
    df['ret_excess'] = df['ret_excess'].fillna(median_target)
    return df

def get_train_valid_data_combined(merged_data, train_idx_merged, val_idx_merged, test_idx_merged, handle_missing_targets=True):
    train_merged, val_merged, test_merged = merged_data.iloc[train_idx_merged], merged_data.iloc[val_idx_merged], merged_data.iloc[test_idx_merged]

    train_permnos = train_merged.index.get_level_values('permno').unique()
    val_permnos = val_merged.index.get_level_values('permno').unique()
    test_permnos = test_merged.index.get_level_values('permno').unique()

    consistent_permnos = train_permnos.intersection(val_permnos).intersection(test_permnos)
    n_shared_permnos = len(consistent_permnos)
    print(f"Shared permnos: {consistent_permnos}")
    print(f"Number of shared permnos: {n_shared_permnos}")

    train_dates = train_merged.index.get_level_values('date').unique()
    val_dates = val_merged.index.get_level_values('date').unique()
    test_dates = test_merged.index.get_level_values('date').unique()

    # Create a complete index for each set
    complete_train_index = pd.MultiIndex.from_product([train_dates, consistent_permnos], names=['date', 'permno'])
    complete_val_index = pd.MultiIndex.from_product([val_dates, consistent_permnos], names=['date', 'permno'])
    complete_test_index = pd.MultiIndex.from_product([test_dates, consistent_permnos], names=['date', 'permno'])

    # Reindex to the complete index to introduce NaNs for missing values
    train_merged = train_merged.reindex(complete_train_index)
    val_merged = val_merged.reindex(complete_val_index)
    test_merged = test_merged.reindex(complete_test_index)

    # Fill missing values using cross-sectional median for narratives
    train_merged = fill_with_cross_sectional_median(train_merged, narratives)
    val_merged = fill_with_cross_sectional_median(val_merged, narratives)
    test_merged = fill_with_cross_sectional_median(test_merged, narratives)

    if handle_missing_targets:
        # Fill target variable with median to handle missing values
        train_merged = fill_target_with_median(train_merged)
        val_merged = fill_target_with_median(val_merged)
        test_merged = fill_target_with_median(test_merged)

    # Sort the index to maintain consistency
    train_merged = train_merged.sort_index()
    val_merged = val_merged.sort_index()
    test_merged = test_merged.sort_index()

    n_train_periods = len(train_dates)
    n_val_periods = len(val_dates)
    n_test_periods = len(test_dates)

    # Reshape X1 inputs
    try:
        X1_train = train_merged[narratives].values.reshape(n_train_periods, n_shared_permnos, -1)
        X1_val = val_merged[narratives].values.reshape(n_val_periods, n_shared_permnos, -1)
        X1_test = test_merged[narratives].values.reshape(n_test_periods, n_shared_permnos, -1)
    except ValueError as e:
        print(f"Error in reshaping arrays: {e}")
        print(f"Train shape: {train_merged[narratives].values.shape}")
        print(f"Validation shape: {val_merged[narratives].values.shape}")
        print(f"Test shape: {test_merged[narratives].values.shape}")
        raise

    # Reshape y and X2 inputs
    y_train = train_merged.ret_excess.unstack('permno')
    y_train = y_train.loc[:, consistent_permnos].values

    y_val = val_merged.ret_excess.unstack('permno')
    y_val = y_val.loc[:, consistent_permnos].values

    y_test = test_merged.ret_excess.unstack('permno')
    y_test = y_test.loc[:, consistent_permnos].values

    X2_train = y_train
    X2_val = y_val
    X2_test = y_test

    return X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test, val_merged, test_merged


**preprocessing code**

In [ ]:

def fill_with_cross_sectional_median(df, narratives):
    medians = df.groupby(level='date')[narratives].transform('median')
    return df.fillna(medians)

def fill_target_with_median(df):
    median_target = df.groupby(level='date')['ret_excess'].transform('median')
    df['ret_excess'] = df['ret_excess'].fillna(median_target)
    return df

def get_train_valid_data_combined(merged_data, train_idx_merged, val_idx_merged, test_idx_merged):
    train_merged, val_merged, test_merged = merged_data.iloc[train_idx_merged], merged_data.iloc[val_idx_merged], merged_data.iloc[test_idx_merged]

    train_permnos = train_merged.index.get_level_values('permno').unique()
    val_permnos = val_merged.index.get_level_values('permno').unique()
    test_permnos = test_merged.index.get_level_values('permno').unique()

    consistent_permnos = train_permnos.intersection(val_permnos).intersection(test_permnos)
    n_shared_permnos = len(consistent_permnos)
    print(f"Shared permnos: {consistent_permnos}")
    print(f"Number of shared permnos: {n_shared_permnos}")

    train_dates = train_merged.index.get_level_values('date').unique()
    val_dates = val_merged.index.get_level_values('date').unique()
    test_dates = test_merged.index.get_level_values('date').unique()

    complete_train_index = pd.MultiIndex.from_product([train_dates, consistent_permnos], names=['date', 'permno'])
    complete_val_index = pd.MultiIndex.from_product([val_dates, consistent_permnos], names=['date', 'permno'])
    complete_test_index = pd.MultiIndex.from_product([test_dates, consistent_permnos], names=['date', 'permno'])

    train_merged = train_merged.reindex(complete_train_index)
    val_merged = val_merged.reindex(complete_val_index)
    test_merged = test_merged.reindex(complete_test_index)

    train_merged = fill_with_cross_sectional_median(train_merged, narratives)
    val_merged = fill_with_cross_sectional_median(val_merged, narratives)
    test_merged = fill_with_cross_sectional_median(test_merged, narratives)

    train_merged = fill_target_with_median(train_merged)
    val_merged = fill_target_with_median(val_merged)
    test_merged = fill_target_with_median(test_merged)

    train_merged = train_merged.sort_index()
    val_merged = val_merged.sort_index()
    test_merged = test_merged.sort_index()

    n_train_periods = len(train_dates)
    n_val_periods = len(val_dates)
    n_test_periods = len(test_dates)

    try:
        X1_train = train_merged[narratives].values.reshape(n_train_periods, n_shared_permnos, -1)
        X1_val = val_merged[narratives].values.reshape(n_val_periods, n_shared_permnos, -1)
        X1_test = test_merged[narratives].values.reshape(n_test_periods, n_shared_permnos, -1)
    except ValueError as e:
        print(f"Error in reshaping arrays: {e}")
        print(f"Train shape: {train_merged[narratives].values.shape}")
        print(f"Validation shape: {val_merged[narratives].values.shape}")
        print(f"Test shape: {test_merged[narratives].values.shape}")
        raise

    # Remove permnos not consistently present in y_train, y_val, and y_test
    y_train = train_merged.ret_excess.unstack('permno')
    y_train = y_train.loc[:, consistent_permnos].values

    y_val = val_merged.ret_excess.unstack('permno')
    y_val = y_val.loc[:, consistent_permnos].values

    y_test = test_merged.ret_excess.unstack('permno')
    y_test = y_test.loc[:, consistent_permnos].values

    X2_train = y_train

    X2_val = y_val

    X2_test = y_test


    return X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test, val_merged, test_merged


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/REVISED_results_CA_2')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)

import time
from tensorflow.keras.callbacks import EarlyStopping
from scipy.optimize import minimize
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import spearmanr
from itertools import product
from tqdm import tqdm
early_stopping = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

cols = ['l1_reg', 'units', 'n_factors', 'fold', 'ic_mean', 'ic_daily_mean', 'ic_daily_std', 'ic_daily_median', 'r2_score', 'mse']

max_epochs = 100
factor_opts = [1,2,3,4,5,6]
l1_reg = [0.00001, 0.0001,0.001, 0.01, 0.1,0.5, 1]
#0.000000001, 0.00000001, 0.0000001, 0.000001,0.00001,0.0001,
param_grid = list(product(l1_reg, factor_opts))
units = 100
batch_size = 32

YEAR = 12

# Use only DateTimeSeriesCV with merged_linear
cv_portfolio = MultipleTimeSeriesCV(n_splits=15, train_period_length=20*YEAR, val_period_length=2*YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)

# Initialize a list to store average test R^2 scores for each factor
average_test_r2_scores_per_factor = []

# Loop over each factor
for n_factors in factor_opts:
    # Initialize a list to store average test R^2 scores from each iteration for the current factor
    average_test_r2_scores = []


    # Create the HDF5 file initially in 'table' format
    with pd.HDFStore(str(results_path / f'scores_{n_factors}.h5'), mode='w', complevel=9, complib='blosc:zstd') as store:
        store.close()

    # Split the data once
    cv_portfolio_splits = list(cv_portfolio.split(merged_df_padded))

    # Store scores for each parameter combination
    for l1_reg_value in l1_reg:
        scores = []
        start = time.time()

        for fold, (train_idx, val_idx, test_idx) in enumerate(cv_portfolio_splits):
            X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test, val_merged, test_merged = get_train_valid_data_combined(merged_df_padded, train_idx, val_idx, test_idx)

            n_tickers = X1_train.shape[1]
            main_model, factor_network_model = make_model(n_tickers=n_tickers,n_factors=n_factors, return_components=True, l1_reg=l1_reg_value)


            # Training the whole model
            main_model.fit([X1_train, X2_train], y_train,
                            batch_size=batch_size,
                            validation_data=([X1_val, X2_val], y_val),
                            epochs=max_epochs,
                            verbose=0,  # Adjust verbosity as needed
                            callbacks=[early_stopping])  # Include both callbacks

            result = pd.DataFrame({'y_pred': main_model.predict([X1_val, X2_val]).reshape(-1),
                                  'y_true': y_val.reshape(-1)},
                                  index=val_merged.index)

            # Calculate spearman's coefficient for val set
            r0 = spearmanr(result.y_true, result.y_pred)[0]

            # Calculate the daily spearman's coefficient
            r1 = result.groupby(level='date').apply(lambda x: spearmanr(x.y_pred, x.y_true)[0])

            # Calculate R^2
            r2 = r2_score(result.y_true, result.y_pred)

            # Calculate MSE
            mse = mean_squared_error(result.y_true, result.y_pred)

            print(f'{time.time()-start:.2f} | factor:{n_factors}| lasso:{l1_reg_value} | fold:{fold:02} | sp-coe: {r0:6.2%} | '
                  f'Avg sp {r1.mean():6.2%} | Median-Sp {r1.median():6.2%} | R^2:{r2:6.2%} | MSE {mse:6.2%}|')

            scores.append([l1_reg_value, units, n_factors, fold, r0, r1.mean(), r1.std(), r1.median(), r2, mse])

        scores_df = pd.DataFrame(scores, columns=cols)
        scores_df.to_hdf(str(results_path / f'scores_{n_factors}.h5'), key=f'{l1_reg_value}/{n_factors}', append=True, format='table')

    print(f" factor {n_factors} training complete.")

    # Find the best performing l1_reg based on the highest average R^2 on the validation set
    scores = []
    with pd.HDFStore(results_path / f'scores_{n_factors}.h5') as store:
        for key in store.keys():
            scores.append(store[key])
    scores = pd.concat(scores)

    # Calculate average validation R^2 for each l1_reg
    avg_r2 = scores.groupby('l1_reg')['r2_score'].mean()

    # Identify the best l1_reg
    best_l1_reg = avg_r2.idxmax()
    print(f"Best l1_reg for factor {n_factors}: {best_l1_reg}")

    # Use the best l1_reg to make predictions on the test set and calculate average R^2 across all folds
    main_model, factor_network_model = make_model(n_tickers=n_tickers,n_factors=n_factors, return_components=True, l1_reg=best_l1_reg)

    test_r2_scores = []
    for fold, (train_idx, val_idx, test_idx) in enumerate(cv_portfolio_splits):
        X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test, val_merged, test_merged = get_train_valid_data_combined(merged_df_padded, train_idx, val_idx, test_idx)

        # Training the whole model with the best l1_reg
        main_model.fit([X1_train, X2_train], y_train,
                        batch_size=batch_size,
                        validation_data=([X1_val, X2_val], y_val),
                        epochs=max_epochs,
                        verbose=0,  # Adjust verbosity as needed
                        callbacks=[early_stopping])  # Include both callbacks

        # Make predictions on the test set
        y_test_pred = main_model.predict([X1_test, X2_test]).reshape(-1)

        # Calculate R^2 for the test set (out-of-sample)
        test_r2 = r2_score(y_test.stack().values, y_test_pred)
        test_r2_scores.append(test_r2)

    # Calculate the average test R^2 across all folds
    average_test_r2 = np.mean(test_r2_scores)
    average_test_r2_scores.append({ 'best_l1_reg': best_l1_reg, 'n_factors': n_factors, 'average_test_r2': average_test_r2})
    print(f"factor {n_factors} Average Test R^2 with best l1_reg {best_l1_reg}: {average_test_r2:.6f}")

# Store the results for the current factor
average_test_r2_scores_per_factor.extend(average_test_r2_scores)

# Convert the list of average test R^2 scores to a DataFrame
average_test_r2_df = pd.DataFrame(average_test_r2_scores_per_factor)

# Save the average test R^2 scores from all iterations to a CSV file
average_test_r2_df.to_csv(results_path / 'average_test_r2_scores_per_factor.csv', index=False)

# Load the saved average R^2 scores
average_test_r2_df = pd.read_csv(results_path / 'average_test_r2_scores_per_factor.csv')

# Calculate the average R^2 for each factor
avg_r2_per_factor = average_test_r2_df.groupby('n_factors')['average_test_r2'].mean().reset_index()

# Display the average R^2 for each factor
print(avg_r2_per_factor)


**More Comprehensive prediction code**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/REVISED_results_CA_2')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)

import time
from tensorflow.keras.callbacks import EarlyStopping
from scipy.optimize import minimize
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import spearmanr
from itertools import product
from tqdm import tqdm
early_stopping = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

cols = ['l1_reg', 'units', 'n_factors', 'fold', 'ic_mean', 'ic_daily_mean', 'ic_daily_std', 'ic_daily_median', 'r2_score', 'mse']

max_epochs = 100
factor_opts = [ 1]
#, 2, 3, 4, 5, 6
l1_reg = [0.0001]
#, 0.001, 0.01, 0.1, 0.5, 1
param_grid = list(product(l1_reg, factor_opts))
units = 100
batch_size = 32

YEAR = 12

# Use only DateTimeSeriesCV with merged_linear
cv_portfolio = MultipleTimeSeriesCV(n_splits=2, train_period_length=20*YEAR, val_period_length=2*YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)

# Initialize a list to store average test R^2 scores for each factor
average_test_r2_scores_per_factor = []

# Dictionary to accumulate predictions by n_factors
accumulated_predictions_by_n_factors = {}

# Loop over each factor
for n_factors in tqdm(factor_opts):
    # Initialize a list to store average test R^2 scores from each iteration for the current factor
    average_test_r2_scores = []

    # Create the HDF5 file initially in 'table' format
    with pd.HDFStore(str(results_path / f'scores_{n_factors}.h5'), mode='w', complevel=9, complib='blosc:zstd') as store:
        store.close()

    # Split the data once
    cv_portfolio_splits = list(cv_portfolio.split(merged_df_padded))

    # Store scores for each parameter combination
    for l1_reg_value in tqdm(l1_reg):
        scores = []
        combined_predictions = []
        start = time.time()

        for fold, (train_idx, val_idx, test_idx) in enumerate(cv_portfolio_splits):
            X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test, val_merged, test_merged = get_train_valid_data_combined(merged_df_padded, train_idx, val_idx, test_idx)

            n_tickers = X1_train.shape[1]
            main_model, factor_network_model = make_model(n_tickers=n_tickers, n_factors=n_factors, return_components=True, l1_reg=l1_reg_value)

            # Training the whole model
            main_model.fit([X1_train, X2_train], y_train,
                           batch_size=batch_size,
                           validation_data=([X1_val, X2_val], y_val),
                           epochs=max_epochs,
                           verbose=0,  # Adjust verbosity as needed
                           callbacks=[early_stopping])  # Include both callbacks


            result = pd.DataFrame({'y_pred': main_model.predict([X1_val, X2_val]).reshape(-1),
                                  'y_true': y_val.reshape(-1)},
                                  index=val_merged.index)

            # Calculate spearman's coefficient for val set
            r0 = spearmanr(result.y_true, result.y_pred)[0]

            # Calculate the daily spearman's coefficient
            r1 = result.groupby(level='date').apply(lambda x: spearmanr(x.y_pred, x.y_true)[0])

            # Calculate R^2
            r2 = r2_score(result.y_true, result.y_pred)

            # Calculate MSE
            mse = mean_squared_error(result.y_true, result.y_pred)

            print(f'{time.time()-start:.2f} | factor:{n_factors}| lasso:{l1_reg_value} | fold:{fold:02} | sp-coe: {r0:6.2%} | '
                  f'Avg sp {r1.mean():6.2%} | Median-Sp {r1.median():6.2%} | R^2:{r2:6.2%} | MSE {mse:6.2%}|')

            scores.append([l1_reg_value, units, n_factors, fold, r0, r1.mean(), r1.std(), r1.median(), r2, mse])

    # Find the best performing l1_reg based on the highest average R^2 on the validation set
    scores = []
    with pd.HDFStore(results_path / f'scores_{n_factors}.h5') as store:
        for key in store.keys():
            scores.append(store[key])
            scores = pd.concat(scores)

            # Calculate average validation R^2 for each l1_reg
            avg_r2 = scores.groupby('l1_reg')['r2_score'].mean()

            # Identify the best l1_reg
            best_l1_reg = avg_r2.idxmax()
            print(f"Best l1_reg for factor {n_factors}: {best_l1_reg}")

    # Use the best l1_reg to make predictions on the test set and calculate average R^2 across all folds
    main_model, factor_network_model = make_model(n_tickers=n_tickers, n_factors=n_factors, return_components=True, l1_reg=best_l1_reg)

    test_r2_scores = []
    combined_predictions = []
    for fold, (train_idx, val_idx, test_idx) in enumerate(cv_portfolio_splits):
        X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test, val_merged, test_merged = get_train_valid_data_combined(merged_df_padded, train_idx, val_idx, test_idx)

        # Training the whole model with the best l1_reg
        main_model.fit([X1_train, X2_train], y_train,
                       batch_size=batch_size,
                       validation_data=([X1_val, X2_val], y_val),
                       epochs=max_epochs,
                       verbose=0,  # Adjust verbosity as needed
                       callbacks=[early_stopping])  # Include both callbacks

        # Make predictions on the test set
        y_test_pred = main_model.predict([X1_test, X2_test]).reshape(-1)

        # Save the test set predictions along with actual test results
        test_result = pd.DataFrame({'y_pred': y_test_pred, 'y_true': y_test.reshape(-1)},
                                   index=test_merged.index)

        # Calculate R^2 for the test set (out-of-sample)
        test_r2 = r2_score(test_result.y_true, test_result.y_pred)
        test_r2_scores.append(test_r2)

        prediction_df = pd.DataFrame({
            'date': test_result.index.get_level_values('date'),
            'permno': test_result.index.get_level_values('permno').astype(str),
            'predicted returns': test_result['y_pred'],
            'actual returns': test_result['y_true']
        }).reset_index(drop=True)  # Reset index to avoid multi-index issues

        # Append current fold's predictions to combined_predictions
        combined_predictions.append(prediction_df)

    # Concatenate predictions from all folds for the current l1_reg and n_factors
    if combined_predictions:
        all_predictions_df = pd.concat(combined_predictions)

        # Check if n_factors key already exists in the dictionary
        if n_factors not in accumulated_predictions_by_n_factors:
            accumulated_predictions_by_n_factors[n_factors] = []

        # Append the combined DataFrame for the current l1_reg to the list for this n_factors
        accumulated_predictions_by_n_factors[n_factors].append(all_predictions_df)

    # Calculate the average test R^2 across all folds
    average_test_r2 = np.mean(test_r2_scores)
    average_test_r2_scores.append({'best_l1_reg': best_l1_reg, 'n_factors': n_factors, 'average_test_r2': average_test_r2})
    print(f"factor {n_factors} Average Test R^2 with best l1_reg {best_l1_reg}: {average_test_r2:.6f}")

# Store the results for the current factor
average_test_r2_scores_per_factor.extend(average_test_r2_scores)

# Convert the list of average test R^2 scores to a DataFrame
average_test_r2_df = pd.DataFrame(average_test_r2_scores_per_factor)

# Save the average test R^2 scores from all iterations to a CSV file
average_test_r2_df.to_csv(results_path / 'average_test_r2_scores_per_factor.csv', index=False)

# After processing all combinations, save the accumulated predictions for each n_factors
for n_factors, df_list in accumulated_predictions_by_n_factors.items():
    # Concatenate all DataFrames for this n_factors
    final_combined_df = pd.concat(df_list).reset_index(drop=True)

    # Save the final combined DataFrame for each n_factors in the HDF5 file
    n_factors_key = f'predictions/{n_factors}'
    with pd.HDFStore(results_path / 'predictions.h5', mode='a', complevel=9, complib='blosc:zstd') as store:
        store.put(n_factors_key, final_combined_df, format='table')
        print(f"Saved combined predictions for n_factors: {n_factors} under key: {n_factors_key}")

print("All predictions have been combined and saved.")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/REVISED_results_CA_2')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True, exist_ok=True)

import time
from tensorflow.keras.callbacks import EarlyStopping
from scipy.optimize import minimize
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import spearmanr
from itertools import product
from tqdm import tqdm


np.random.seed(42)



early_stopping = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

cols = ['l1_reg', 'units', 'n_factors', 'fold', 'ic_mean', 'ic_daily_mean', 'ic_daily_std', 'ic_daily_median', 'r2_score', 'mse']

max_epochs = 100
factor_opts = [1,2,3,4,5,6]
l1_reg = [0.00001,0.0001, 0.001, 0.01, 0.1, 0.5, 1]
#, 0.001, 0.01, 0.1, 0.5, 1
param_grid = list(product(l1_reg, factor_opts))
units = 100
batch_size = 32

YEAR = 12

# Use only DateTimeSeriesCV with merged_linear
cv_portfolio = MultipleTimeSeriesCV(n_splits=15, train_period_length=20*YEAR, val_period_length=2*YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)

# Initialize a list to store average test R^2 scores for each factor
average_test_r2_scores_per_factor = []

# Dictionary to accumulate predictions by n_factors
accumulated_predictions_by_n_factors = {}

# Ensure merged_df_padded is defined and loaded
# merged_df_padded = pd.read_csv('your_data.csv') # example

# Loop over each factor
for n_factors in tqdm(factor_opts):
    # Initialize a list to store average test R^2 scores from each iteration for the current factor
    average_test_r2_scores = []

    # Create the HDF5 file initially in 'table' format
    with pd.HDFStore(str(results_path / f'scores_{n_factors}.h5'), mode='w', complevel=9, complib='blosc:zstd') as store:
        store.close()

    # Split the data once
    cv_portfolio_splits = list(cv_portfolio.split(merged_df_padded))

    # Store scores for each parameter combination
    for l1_reg_value in tqdm(l1_reg):
        scores = []
        combined_predictions = []
        start = time.time()

        for fold, (train_idx, val_idx, test_idx) in enumerate(cv_portfolio_splits):
            X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test, val_merged, test_merged = get_train_valid_data_combined(merged_df_padded, train_idx, val_idx, test_idx)

            n_tickers = X1_train.shape[1]
            main_model, factor_network_model = make_model(n_tickers=n_tickers, n_factors=n_factors, return_components=True, l1_reg=l1_reg_value)

            # Training the whole model
            main_model.fit([X1_train, X2_train], y_train,
                           batch_size=batch_size,
                           validation_data=([X1_val, X2_val], y_val),
                           epochs=max_epochs,
                           verbose=0,  # Adjust verbosity as needed
                           callbacks=[early_stopping])

            result = pd.DataFrame({'y_pred': main_model.predict([X1_val, X2_val]).reshape(-1),
                                  'y_true': y_val.reshape(-1)},
                                  index=val_merged.index)

            # Calculate spearman's coefficient for val set
            r0 = spearmanr(result.y_true, result.y_pred)[0]

            # Calculate the daily spearman's coefficient
            r1 = result.groupby(level='date').apply(lambda x: spearmanr(x.y_pred, x.y_true)[0])

            # Calculate R^2
            r2 = r2_score(result.y_true, result.y_pred)

            # Calculate MSE
            mse = mean_squared_error(result.y_true, result.y_pred)

            print(f'{time.time()-start:.2f} | factor:{n_factors}| lasso:{l1_reg_value} | fold:{fold:02} | sp-coe: {r0:6.2%} | '
                  f'Avg sp {r1.mean():6.2%} | Median-Sp {r1.median():6.2%} | R^2:{r2:6.2%} | MSE {mse:6.2%}|')

            scores.append([l1_reg_value, units, n_factors, fold, r0, r1.mean(), r1.std(), r1.median(), r2, mse])

        # Save scores to the HDF5 file
        scores_df = pd.DataFrame(scores, columns=cols)
        with pd.HDFStore(str(results_path / f'scores_{n_factors}.h5'), mode='a', complevel=9, complib='blosc:zstd') as store:
            store.put(f'scores_{l1_reg_value}', scores_df, format='table')

    # Find the best performing l1_reg based on the highest average R^2 on the validation set
    with pd.HDFStore(str(results_path / f'scores_{n_factors}.h5')) as store:
        scores = [store[key] for key in store.keys()]
        scores = pd.concat(scores)

    # Calculate average validation R^2 for each l1_reg
    avg_r2 = scores.groupby('l1_reg')['r2_score'].mean()

    # Identify the best l1_reg
    best_l1_reg = avg_r2.idxmax()
    print(f"Best l1_reg for factor {n_factors}: {best_l1_reg}")


    test_r2_scores = []
    combined_predictions = []
    for fold, (train_idx, val_idx, test_idx) in enumerate(cv_portfolio_splits):
        X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test, val_merged, test_merged = get_train_valid_data_combined(merged_df_padded, train_idx, val_idx, test_idx)

        n_tickers = X1_train.shape[1]

        # Use the best l1_reg to make predictions on the test set and calculate average R^2 across all folds
        main_model, factor_network_model = make_model(n_tickers=n_tickers, n_factors=n_factors, return_components=True, l1_reg=best_l1_reg)


        # Training the whole model with the best l1_reg
        main_model.fit([X1_train, X2_train], y_train,
                       batch_size=batch_size,
                       validation_data=([X1_val, X2_val], y_val),
                       epochs=max_epochs,
                       verbose=0,  # Adjust verbosity as needed
                       callbacks=[early_stopping])

        # Make predictions on the test set
        y_test_pred = main_model.predict([X1_test, X2_test]).reshape(-1)

        # Save the test set predictions along with actual test results and best l1_reg
        test_result = pd.DataFrame({'y_pred': y_test_pred, 'y_true': y_test.reshape(-1), 'best_l1_reg': best_l1_reg},
                                   index=test_merged.index)

        # Calculate R^2 for the test set (out-of-sample)
        test_r2 = r2_score(test_result.y_true, test_result.y_pred)
        test_r2_scores.append(test_r2)

        prediction_df = pd.DataFrame({
            'date': test_result.index.get_level_values('date'),
            'permno': test_result.index.get_level_values('permno').astype(str),
            'predicted returns': test_result['y_pred'],
            'actual returns': test_result['y_true'],
            'best_l1_reg': best_l1_reg
        }).reset_index(drop=True)

        # Append current fold's predictions to combined_predictions
        combined_predictions.append(prediction_df)

    # Concatenate predictions from all folds for the current l1_reg and n_factors
    if combined_predictions:
        all_predictions_df = pd.concat(combined_predictions)

        # Check if n_factors key already exists in the dictionary
        if n_factors not in accumulated_predictions_by_n_factors:
            accumulated_predictions_by_n_factors[n_factors] = []

        # Append the combined DataFrame for the current l1_reg to the list for this n_factors
        accumulated_predictions_by_n_factors[n_factors].append(all_predictions_df)

    # Calculate the average test R^2 across all folds
    average_test_r2 = np.mean(test_r2_scores)
    average_test_r2_scores.append({'best_l1_reg': best_l1_reg, 'n_factors': n_factors, 'average_test_r2': average_test_r2})
    print(f"factor {n_factors} Average Test R^2 with best l1_reg {best_l1_reg}: {average_test_r2:.6f}")

# Store the results for the current factor
average_test_r2_scores_per_factor.extend(average_test_r2_scores)

# Convert the list of average test R^2 scores to a DataFrame
average_test_r2_df = pd.DataFrame(average_test_r2_scores_per_factor)

# Save the average test R^2 scores from all iterations to a CSV file
average_test_r2_df.to_csv(results_path / 'average_test_r2_scores_per_factor.csv', index=False)

# After processing all combinations, save the accumulated predictions for each n_factors
for n_factors, df_list in accumulated_predictions_by_n_factors.items():
    # Concatenate all DataFrames for this n_factors
    final_combined_df = pd.concat(df_list).reset_index(drop=True)

    # Save the final combined DataFrame for each n_factors in the HDF5 file
    n_factors_key = f'predictions/{n_factors}'
    with pd.HDFStore(results_path / 'predictions.h5', mode='a', complevel=9, complib='blosc:zstd') as store:
        store.put(n_factors_key, final_combined_df, format='table')
        print(f"Saved combined predictions for n_factors: {n_factors} under key: {n_factors_key}")

print("All predictions have been combined and saved.")
